In [12]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Regression Models
from sklearn.linear_model import LinearRegression
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)

In [14]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [15]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [16]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [17]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [18]:
# Display Feature Selection Info
print('=== FEATURE SELECTION INFO ===')
print(f'Tipe: {type(feat_select)}')
print(f'\nIsi: ')
if isinstance(feat_select, dict):
    print(feat_select)
elif isinstance(feat_select, list):
    print(f'Jumlah fitur: {len(feat_select)}')
    print(feat_select)
else:
    print(feat_select)

=== FEATURE SELECTION INFO ===
Tipe: <class 'dict'>

Isi: 
{'kolom_id': ['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], 'kolom_kat': ['provinsi', 'pulau', 'tipe_wilayah'], 'target': 'produktivitas_kuha', 'kolom_num_semua': ['luas_panen_ha', 'PRECTOTCORR_sum_tahun', 'PRECTOTCORR_mean_tahun', 'PRECTOTCORR_max_tahun', 'PRECTOTCORR_std_tahun', 'T2M_mean_tahun', 'T2M_min_tahun', 'T2M_max_tahun', 'T2M_std_tahun', 'RH2M_mean_tahun', 'RH2M_min_tahun', 'RH2M_std_tahun', 'jumlah_hari_hujan_tahun', 'jumlah_hari_kering_tahun', 'jumlah_hari_hujan_ekstrem', 'PRECTOTCORR_sum_MH', 'PRECTOTCORR_mean_MH', 'PRECTOTCORR_max_MH', 'T2M_mean_MH', 'T2M_std_MH', 'RH2M_mean_MH', 'jumlah_hari_hujan_MH', 'jumlah_hari_kering_MH', 'max_consecutive_dry_MH', 'PRECTOTCORR_sum_MK', 'PRECTOTCORR_mean_MK', 'PRECTOTCORR_max_MK', 'T2M_mean_MK', 'T2M_std_MK', 'RH2M_mean_MK', 'jumlah_hari_hujan_MK', 'jumlah_hari_kering_MK', 'max_consecutive_dry_MK', 'PRECTOTCORR_sum_tahun_lag1', 'PRECTOTCORR_sum_MH_lag1', 'PRECTOTCORR_su

In [19]:
TARGET = feat_select['target']

In [20]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [21]:
print('=== VERIFIKASI LOAD DATA ===')
print(f'X_train : {X_train.shape}')
print(f'X_val : {X_val.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train : {y_train_arr.shape}')
print(f'y_val : {y_val_arr.shape}')
print(f'y_test : {y_test_arr.shape}')
print(f'\nTotal fitur: {X_train.shape[1]}')
print(f'Target : {TARGET}')
print(f'\nNull check:')
print(f' X_train: {X_train.isnull().sum().sum()}')
print(f' X_val : {X_val.isnull().sum().sum()}')
print(f' X_test : {X_test.isnull().sum().sum()}')
print(f'\nRange target:')
print(f' Train: {y_train_arr.min():.2f} - {y_train_arr.max():.2f} Qu/Ha')
print(f' Val : {y_val_arr.min():.2f} - {y_val_arr.max():.2f} Qu/Ha')
print(f' Test : {y_test_arr.min():.2f} - {y_test_arr.max():.2f} Qu/Ha')
print(f'\nArtifak loaded:')
print(f' minmax_scaler : OK')
print(f' kolom_info : {len(feat_select)} entri')
print(f' winsor_bounds : {len(winsor)} kolom')
# print(f' feature_selection : drop {len(feat_select["kolom_drop_corr"])} kolom high-corr')
print(f'\nFolder output:')
print(f' Model : {MODEL_FOLDER}')
print(f' Gambar : {EDA_FOLDER}')
print('\nData siap. Lanjut ke modeling.')

=== VERIFIKASI LOAD DATA ===
X_train : (2251, 68)
X_val : (450, 68)
X_test : (893, 68)
y_train : (2251,)
y_val : (450,)
y_test : (893,)

Total fitur: 68
Target : produktivitas_kuha

Null check:
 X_train: 0
 X_val : 0
 X_test : 0

Range target:
 Train: 12.32 - 75.54 Qu/Ha
 Val : 18.63 - 71.66 Qu/Ha
 Test : 12.54 - 75.32 Qu/Ha

Artifak loaded:
 minmax_scaler : OK
 kolom_info : 22 entri
 winsor_bounds : 27 kolom

Folder output:
 Model : dataset/model_outputs
 Gambar : dataset/eda_outputs

Data siap. Lanjut ke modeling.


In [22]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [23]:
print("Mulai training Linear Regression")

# inisialisasi model
lr_model = LinearRegression()

# training model
lr_model.fit(X_train, y_train)

print("Training selesai")

Mulai training Linear Regression
Training selesai


In [24]:
# Prediksi
train_pred = lr_model.predict(X_train)
val_pred = lr_model.predict(X_val)
test_pred = lr_model.predict(X_test)

In [25]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [26]:
# evaluasi train
train_mae, train_rmse, train_r2 = hitung_metrics(
    y_train,
    train_pred
)

In [27]:
# evaluasi validation
val_mae, val_rmse, val_r2 = hitung_metrics(
    y_val,
    val_pred
)

In [28]:
# evaluasi test
test_mae, test_rmse, test_r2 = hitung_metrics(
    y_test,
    test_pred
)

In [29]:
print("HASIL LINEAR REGRESSION")

print("\nTrain")
print(f"MAE  : {train_mae:.4f}")
print(f"RMSE : {train_rmse:.4f}")
print(f"R2   : {train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R2   : {val_r2:.4f}")

print("\nTest")
print(f"MAE  : {test_mae:.4f}")
print(f"RMSE : {test_rmse:.4f}")
print(f"R2   : {test_r2:.4f}")

HASIL LINEAR REGRESSION

Train
MAE  : 4.6277
RMSE : 5.9429
R2   : 0.6935

Validation
MAE  : 4.6548
RMSE : 6.0211
R2   : 0.6576

Test
MAE  : 4.8942
RMSE : 6.2934
R2   : 0.6447


In [30]:
# Simpan Model Linear Regression
joblib.dump(
    lr_model,
    os.path.join(BASE_PATH, 'models', 'linear_regression.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [31]:
# Simpan metrics
metrics_lr = pd.DataFrame({
    "model": ["Linear Regression"],
    "train_mae": [train_mae],
    "train_rmse": [train_rmse],
    "train_r2": [train_r2],
    "val_mae": [val_mae],
    "val_rmse": [val_rmse],
    "val_r2": [val_r2],
    "test_mae": [test_mae],
    "test_rmse": [test_rmse],
    "test_r2": [test_r2]
})

os.makedirs("results", exist_ok=True)

metrics_lr.to_csv(
    os.path.join(BASE_PATH, "results", "linear_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [32]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_lr"] = test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "linear_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  prediction_lr
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82      49.273620
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53      51.262015
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15      50.869307
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77      53.274884
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82      49.671488


In [ ]:
joblib.dump(
    lr_model,
    os.path.join(
        MODEL_FOLDER,
        'linear_regression_model.joblib'
    )
)

print("Linear Regression model berhasil disimpan.")

KMeans model berhasil disimpan.
